In [3]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
import json

# 클로드 개별 업무 지시용 함수
def run_prompt(test_case):
    """프롬프트와 테스트 케이스 입력값을 병합한 후 결과를 반환"""
    prompt = f"""
    다음 작업을 수행하라:

    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

# 클로드 개별 항목 검토 및 채점용 함수
def run_test_case(test_case):
    """run_prompt를 호출한 후 결과를 채점"""
    output = run_prompt(test_case)

    # 채점로직(추후 구현 예정으로 현재는 10점으로 하드코딩)
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }
# 위 두 개의 함수를 활용하여 수행한 작업 및 이에 대한 채점 결과를 순차적으로 제시
def run_eval(dataset):
    """데이터셋을 로드하고 각 테스트 케이스에 run_test_case를 호출"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
# 결과값을 unicode가 아닌 한글로 인출하기 위해 ensure_ascii=False를 추가
json.dumps(results, indent=2, ensure_ascii=False)


'[\n  {\n    "output": "# AWS S3에서 특정 날짜의 객체를 필터링하는 Python 함수\\n\\n다음은 boto3를 사용하여 2024년 1월에 업로드된 S3 객체를 필터링하는 완전한 구현입니다.\\n\\n## 1. 기본 구현\\n\\n```python\\nimport boto3\\nfrom datetime import datetime, timedelta\\nfrom typing import List, Dict\\n\\ndef filter_s3_objects_by_month(bucket_name: str, year: int, month: int) -> List[Dict]:\\n    \\"\\"\\"\\n    S3 버킷에서 특정 월에 업로드된 객체를 필터링합니다.\\n    \\n    Args:\\n        bucket_name: S3 버킷 이름\\n        year: 년도 (예: 2024)\\n        month: 월 (예: 1)\\n    \\n    Returns:\\n        필터링된 객체 정보 리스트\\n    \\"\\"\\"\\n    s3_client = boto3.client(\'s3\')\\n    filtered_objects = []\\n    \\n    # 해당 월의 시작일과 마지막 날 계산\\n    start_date = datetime(year, month, 1)\\n    \\n    # 다음 월의 1일을 구한 후 1일 전을 마지막 날로 설정\\n    if month == 12:\\n        end_date = datetime(year + 1, 1, 1)\\n    else:\\n        end_date = datetime(year, month + 1, 1)\\n    end_date = end_date - timedelta(days=1)\\n    \\n    print(f\\"검색 기간: {start_date.date()} ~ {end_date.date()}\\")\

In [11]:
from IPython.display import Markdown
Markdown(results[0]["output"])

# AWS S3에서 특정 날짜의 객체를 필터링하는 Python 함수

다음은 boto3를 사용하여 2024년 1월에 업로드된 S3 객체를 필터링하는 완전한 구현입니다.

## 1. 기본 구현

```python
import boto3
from datetime import datetime, timedelta
from typing import List, Dict

def filter_s3_objects_by_month(bucket_name: str, year: int, month: int) -> List[Dict]:
    """
    S3 버킷에서 특정 월에 업로드된 객체를 필터링합니다.
    
    Args:
        bucket_name: S3 버킷 이름
        year: 년도 (예: 2024)
        month: 월 (예: 1)
    
    Returns:
        필터링된 객체 정보 리스트
    """
    s3_client = boto3.client('s3')
    filtered_objects = []
    
    # 해당 월의 시작일과 마지막 날 계산
    start_date = datetime(year, month, 1)
    
    # 다음 월의 1일을 구한 후 1일 전을 마지막 날로 설정
    if month == 12:
        end_date = datetime(year + 1, 1, 1)
    else:
        end_date = datetime(year, month + 1, 1)
    end_date = end_date - timedelta(days=1)
    
    print(f"검색 기간: {start_date.date()} ~ {end_date.date()}")
    
    try:
        # S3 리스트 객체 (페이지네이션 지원)
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket_name)
        
        for page in pages:
            if 'Contents' not in page:
                continue
            
            for obj in page['Contents']:
                last_modified = obj['LastModified']
                
                # 날짜 비교 (timezone aware 처리)
                if start_date <= last_modified <= end_date.replace(hour=23, minute=59, second=59):
                    filtered_objects.append({
                        'Key': obj['Key'],
                        'LastModified': last_modified,
                        'Size': obj['Size'],
                        'StorageClass': obj['StorageClass']
                    })
        
        print(f"\n✓ 검색 완료: {len(filtered_objects)}개의 객체 발견")
        return filtered_objects
    
    except Exception as e:
        print(f"✗ 오류 발생: {str(e)}")
        return []


# 사용 예시
if __name__ == "__main__":
    bucket_name = "my-s3-bucket"
    
    # 2024년 1월 객체 필터링
    result = filter_s3_objects_by_month(bucket_name, 2024, 1)
    
    for obj in result:
        print(f"파일명: {obj['Key']}")
        print(f"수정일: {obj['LastModified']}")
        print(f"크기: {obj['Size']} bytes")
        print("-" * 50)
```

## 2. 고급 구현 (날짜 범위 지정)

```python
import boto3
from datetime import datetime
from typing import List, Dict, Optional

def filter_s3_objects_by_date_range(
    bucket_name: str,
    start_date: datetime,
    end_date: datetime,
    prefix: str = "",
    include_file_extensions: Optional[List[str]]